In [1]:
import pandas as pd
import numpy as np

In [6]:
# lowercase, price to number, and date text to date
df = pd.read_csv("../Model_Data/pricing_history_extracted.csv")
df["ctx"] = df.context.str.lower()
df["price_num"] = df.price.str.replace("$","",regex=False).str.replace(",","",regex=False).astype(float)
df["date"] = pd.to_datetime(df.snapshot_date, errors="coerce")
df

,provider,snapshot_date,price,context,source_url,ctx,price_num,date
0,openai,2023-02-27,$0.0004,tokens is about 750 words. this paragraph is ...,https://web.archive.org/web/20230227230602/htt...,tokens is about 750 words. this paragraph is ...,0.0004,2023-02-27
1,openai,2023-02-27,$0.005,this paragraph is 35 tokens. learn more ada fa...,https://web.archive.org/web/20230227230602/htt...,this paragraph is 35 tokens. learn more ada fa...,0.0050,2023-02-27
2,openai,2023-02-27,$0.0020,ns. learn more ada fastest $0.0004 / 1k tokens...,https://web.archive.org/web/20230227230602/htt...,ns. learn more ada fastest $0.0004 / 1k tokens...,0.0020,2023-02-27
3,openai,2023-02-27,$0.0200,kens babbage $0.005 / 1k tokens curie $0.0020 ...,https://web.archive.org/web/20230227230602/htt...,kens babbage $0.005 / 1k tokens curie $0.0020 ...,0.0200,2023-02-27
4,openai,2023-02-27,$0.0004,n requests to that model. learn more about fin...,https://web.archive.org/web/20230227230602/htt...,n requests to that model. learn more about fin...,0.0004,2023-02-27
...,...,...,...,...,...,...,...,...
3277,google_vertex,2026-04-01,$2.00,tokens (or $0.0005/page) mistral medium 3 inpu...,https://web.archive.org/web/20260401044055/htt...,tokens (or $0.0005/page) mistral medium 3 inpu...,2.0000,2026-04-01
3278,google_vertex,2026-04-01,$0.10,million tokens output: $2.00 / million tokens...,https://web.archive.org/web/20260401044055/htt...,million tokens output: $2.00 / million tokens...,0.1000,2026-04-01
3279,google_vertex,2026-04-01,$0.30,million tokens mistral small 3.1 (25.03) inpu...,https://web.archive.org/web/20260401044055/htt...,million tokens mistral small 3.1 (25.03) inpu...,0.3000,2026-04-01
3280,google_vertex,2026-04-01,$0.30,input: $0.10 / million tokens output: $0.30 / ...,https://web.archive.org/web/20260401044055/htt...,input: $0.10 / million tokens output: $0.30 / ...,0.3000,2026-04-01


In [7]:
# assign price to model
OAI_MODELS = ["gpt-4o-mini","gpt-4o","gpt-4-turbo","gpt-4-32k","gpt-4-vision","gpt-4", "gpt-3.5-turbo-16k","gpt-3.5-turbo","davinci","curie","babbage","ada"]

def nearest_model(row):
    ctx, price = row.ctx, row.price.lower()
    i = ctx.find(price); prefix = ctx[:i] if i > 0 else ctx
    best, bp = None, -1
    for m in OAI_MODELS:                
        p = prefix.rfind(m)              
        if p > bp: best, bp = m, p
    return best

df

,provider,snapshot_date,price,context,source_url,ctx,price_num,date
0,openai,2023-02-27,$0.0004,tokens is about 750 words. this paragraph is ...,https://web.archive.org/web/20230227230602/htt...,tokens is about 750 words. this paragraph is ...,0.0004,2023-02-27
1,openai,2023-02-27,$0.005,this paragraph is 35 tokens. learn more ada fa...,https://web.archive.org/web/20230227230602/htt...,this paragraph is 35 tokens. learn more ada fa...,0.0050,2023-02-27
2,openai,2023-02-27,$0.0020,ns. learn more ada fastest $0.0004 / 1k tokens...,https://web.archive.org/web/20230227230602/htt...,ns. learn more ada fastest $0.0004 / 1k tokens...,0.0020,2023-02-27
3,openai,2023-02-27,$0.0200,kens babbage $0.005 / 1k tokens curie $0.0020 ...,https://web.archive.org/web/20230227230602/htt...,kens babbage $0.005 / 1k tokens curie $0.0020 ...,0.0200,2023-02-27
4,openai,2023-02-27,$0.0004,n requests to that model. learn more about fin...,https://web.archive.org/web/20230227230602/htt...,n requests to that model. learn more about fin...,0.0004,2023-02-27
...,...,...,...,...,...,...,...,...
3277,google_vertex,2026-04-01,$2.00,tokens (or $0.0005/page) mistral medium 3 inpu...,https://web.archive.org/web/20260401044055/htt...,tokens (or $0.0005/page) mistral medium 3 inpu...,2.0000,2026-04-01
3278,google_vertex,2026-04-01,$0.10,million tokens output: $2.00 / million tokens...,https://web.archive.org/web/20260401044055/htt...,million tokens output: $2.00 / million tokens...,0.1000,2026-04-01
3279,google_vertex,2026-04-01,$0.30,million tokens mistral small 3.1 (25.03) inpu...,https://web.archive.org/web/20260401044055/htt...,million tokens mistral small 3.1 (25.03) inpu...,0.3000,2026-04-01
3280,google_vertex,2026-04-01,$0.30,input: $0.10 / million tokens output: $0.30 / ...,https://web.archive.org/web/20260401044055/htt...,input: $0.10 / million tokens output: $0.30 / ...,0.3000,2026-04-01


In [8]:
# filters data
oai = df[df.provider.isin(["openai","openai_api"])].copy()
oai = oai[oai.ctx.str.contains("1k tokens|1m tokens|/ 1k|/ 1m", regex=True)]
oai["model"] = oai.apply(nearest_model, axis=1)
oai = oai.dropna(subset=["model"])
df

,provider,snapshot_date,price,context,source_url,ctx,price_num,date
0,openai,2023-02-27,$0.0004,tokens is about 750 words. this paragraph is ...,https://web.archive.org/web/20230227230602/htt...,tokens is about 750 words. this paragraph is ...,0.0004,2023-02-27
1,openai,2023-02-27,$0.005,this paragraph is 35 tokens. learn more ada fa...,https://web.archive.org/web/20230227230602/htt...,this paragraph is 35 tokens. learn more ada fa...,0.0050,2023-02-27
2,openai,2023-02-27,$0.0020,ns. learn more ada fastest $0.0004 / 1k tokens...,https://web.archive.org/web/20230227230602/htt...,ns. learn more ada fastest $0.0004 / 1k tokens...,0.0020,2023-02-27
3,openai,2023-02-27,$0.0200,kens babbage $0.005 / 1k tokens curie $0.0020 ...,https://web.archive.org/web/20230227230602/htt...,kens babbage $0.005 / 1k tokens curie $0.0020 ...,0.0200,2023-02-27
4,openai,2023-02-27,$0.0004,n requests to that model. learn more about fin...,https://web.archive.org/web/20230227230602/htt...,n requests to that model. learn more about fin...,0.0004,2023-02-27
...,...,...,...,...,...,...,...,...
3277,google_vertex,2026-04-01,$2.00,tokens (or $0.0005/page) mistral medium 3 inpu...,https://web.archive.org/web/20260401044055/htt...,tokens (or $0.0005/page) mistral medium 3 inpu...,2.0000,2026-04-01
3278,google_vertex,2026-04-01,$0.10,million tokens output: $2.00 / million tokens...,https://web.archive.org/web/20260401044055/htt...,million tokens output: $2.00 / million tokens...,0.1000,2026-04-01
3279,google_vertex,2026-04-01,$0.30,million tokens mistral small 3.1 (25.03) inpu...,https://web.archive.org/web/20260401044055/htt...,million tokens mistral small 3.1 (25.03) inpu...,0.3000,2026-04-01
3280,google_vertex,2026-04-01,$0.30,input: $0.10 / million tokens output: $0.30 / ...,https://web.archive.org/web/20260401044055/htt...,input: $0.10 / million tokens output: $0.30 / ...,0.3000,2026-04-01


In [9]:
# normalize data from 1k to 1m, remove enterprise subcription version 
oai["per_1m"] = np.where(oai.ctx.str.contains("1k"), oai.price_num*1000, oai.price_num)
oai = oai[(oai.per_1m >= 0.05) & (oai.per_1m <= 200)]      
oai = oai.drop_duplicates(["model","date","per_1m"])
df

,provider,snapshot_date,price,context,source_url,ctx,price_num,date
0,openai,2023-02-27,$0.0004,tokens is about 750 words. this paragraph is ...,https://web.archive.org/web/20230227230602/htt...,tokens is about 750 words. this paragraph is ...,0.0004,2023-02-27
1,openai,2023-02-27,$0.005,this paragraph is 35 tokens. learn more ada fa...,https://web.archive.org/web/20230227230602/htt...,this paragraph is 35 tokens. learn more ada fa...,0.0050,2023-02-27
2,openai,2023-02-27,$0.0020,ns. learn more ada fastest $0.0004 / 1k tokens...,https://web.archive.org/web/20230227230602/htt...,ns. learn more ada fastest $0.0004 / 1k tokens...,0.0020,2023-02-27
3,openai,2023-02-27,$0.0200,kens babbage $0.005 / 1k tokens curie $0.0020 ...,https://web.archive.org/web/20230227230602/htt...,kens babbage $0.005 / 1k tokens curie $0.0020 ...,0.0200,2023-02-27
4,openai,2023-02-27,$0.0004,n requests to that model. learn more about fin...,https://web.archive.org/web/20230227230602/htt...,n requests to that model. learn more about fin...,0.0004,2023-02-27
...,...,...,...,...,...,...,...,...
3277,google_vertex,2026-04-01,$2.00,tokens (or $0.0005/page) mistral medium 3 inpu...,https://web.archive.org/web/20260401044055/htt...,tokens (or $0.0005/page) mistral medium 3 inpu...,2.0000,2026-04-01
3278,google_vertex,2026-04-01,$0.10,million tokens output: $2.00 / million tokens...,https://web.archive.org/web/20260401044055/htt...,million tokens output: $2.00 / million tokens...,0.1000,2026-04-01
3279,google_vertex,2026-04-01,$0.30,million tokens mistral small 3.1 (25.03) inpu...,https://web.archive.org/web/20260401044055/htt...,million tokens mistral small 3.1 (25.03) inpu...,0.3000,2026-04-01
3280,google_vertex,2026-04-01,$0.30,input: $0.10 / million tokens output: $0.30 / ...,https://web.archive.org/web/20260401044055/htt...,input: $0.10 / million tokens output: $0.30 / ...,0.3000,2026-04-01


In [10]:
# low =input, high = output
def assign(g):                                              
    v = sorted(g.per_1m.unique())
    return pd.Series({"input_price_per_1m": v[0], "output_price_per_1m": v[-1]})

openai_clean = oai.groupby(["model","date"]).apply(assign, include_groups=False).reset_index()
openai_clean["provider"] = "openai"   
df

,provider,snapshot_date,price,context,source_url,ctx,price_num,date
0,openai,2023-02-27,$0.0004,tokens is about 750 words. this paragraph is ...,https://web.archive.org/web/20230227230602/htt...,tokens is about 750 words. this paragraph is ...,0.0004,2023-02-27
1,openai,2023-02-27,$0.005,this paragraph is 35 tokens. learn more ada fa...,https://web.archive.org/web/20230227230602/htt...,this paragraph is 35 tokens. learn more ada fa...,0.0050,2023-02-27
2,openai,2023-02-27,$0.0020,ns. learn more ada fastest $0.0004 / 1k tokens...,https://web.archive.org/web/20230227230602/htt...,ns. learn more ada fastest $0.0004 / 1k tokens...,0.0020,2023-02-27
3,openai,2023-02-27,$0.0200,kens babbage $0.005 / 1k tokens curie $0.0020 ...,https://web.archive.org/web/20230227230602/htt...,kens babbage $0.005 / 1k tokens curie $0.0020 ...,0.0200,2023-02-27
4,openai,2023-02-27,$0.0004,n requests to that model. learn more about fin...,https://web.archive.org/web/20230227230602/htt...,n requests to that model. learn more about fin...,0.0004,2023-02-27
...,...,...,...,...,...,...,...,...
3277,google_vertex,2026-04-01,$2.00,tokens (or $0.0005/page) mistral medium 3 inpu...,https://web.archive.org/web/20260401044055/htt...,tokens (or $0.0005/page) mistral medium 3 inpu...,2.0000,2026-04-01
3278,google_vertex,2026-04-01,$0.10,million tokens output: $2.00 / million tokens...,https://web.archive.org/web/20260401044055/htt...,million tokens output: $2.00 / million tokens...,0.1000,2026-04-01
3279,google_vertex,2026-04-01,$0.30,million tokens mistral small 3.1 (25.03) inpu...,https://web.archive.org/web/20260401044055/htt...,million tokens mistral small 3.1 (25.03) inpu...,0.3000,2026-04-01
3280,google_vertex,2026-04-01,$0.30,input: $0.10 / million tokens output: $0.30 / ...,https://web.archive.org/web/20260401044055/htt...,input: $0.10 / million tokens output: $0.30 / ...,0.3000,2026-04-01


In [11]:
print(openai_clean[openai_clean.model=="gpt-4"])
df

    model       date  input_price_per_1m  output_price_per_1m provider
54  gpt-4 2023-10-03               30.00                 60.0   openai
55  gpt-4 2024-01-01               10.00                 60.0   openai
56  gpt-4 2024-10-01               10.00                 60.0   openai
57  gpt-4 2025-01-09               10.00                 60.0   openai
58  gpt-4 2025-07-02                0.05                  3.0   openai
59  gpt-4 2025-10-04                0.05                  3.0   openai
60  gpt-4 2026-01-02                0.05                  3.0   openai


,provider,snapshot_date,price,context,source_url,ctx,price_num,date
0,openai,2023-02-27,$0.0004,tokens is about 750 words. this paragraph is ...,https://web.archive.org/web/20230227230602/htt...,tokens is about 750 words. this paragraph is ...,0.0004,2023-02-27
1,openai,2023-02-27,$0.005,this paragraph is 35 tokens. learn more ada fa...,https://web.archive.org/web/20230227230602/htt...,this paragraph is 35 tokens. learn more ada fa...,0.0050,2023-02-27
2,openai,2023-02-27,$0.0020,ns. learn more ada fastest $0.0004 / 1k tokens...,https://web.archive.org/web/20230227230602/htt...,ns. learn more ada fastest $0.0004 / 1k tokens...,0.0020,2023-02-27
3,openai,2023-02-27,$0.0200,kens babbage $0.005 / 1k tokens curie $0.0020 ...,https://web.archive.org/web/20230227230602/htt...,kens babbage $0.005 / 1k tokens curie $0.0020 ...,0.0200,2023-02-27
4,openai,2023-02-27,$0.0004,n requests to that model. learn more about fin...,https://web.archive.org/web/20230227230602/htt...,n requests to that model. learn more about fin...,0.0004,2023-02-27
...,...,...,...,...,...,...,...,...
3277,google_vertex,2026-04-01,$2.00,tokens (or $0.0005/page) mistral medium 3 inpu...,https://web.archive.org/web/20260401044055/htt...,tokens (or $0.0005/page) mistral medium 3 inpu...,2.0000,2026-04-01
3278,google_vertex,2026-04-01,$0.10,million tokens output: $2.00 / million tokens...,https://web.archive.org/web/20260401044055/htt...,million tokens output: $2.00 / million tokens...,0.1000,2026-04-01
3279,google_vertex,2026-04-01,$0.30,million tokens mistral small 3.1 (25.03) inpu...,https://web.archive.org/web/20260401044055/htt...,million tokens mistral small 3.1 (25.03) inpu...,0.3000,2026-04-01
3280,google_vertex,2026-04-01,$0.30,input: $0.10 / million tokens output: $0.30 / ...,https://web.archive.org/web/20260401044055/htt...,input: $0.10 / million tokens output: $0.30 / ...,0.3000,2026-04-01
